In [ ]:
from common import *

## PRILOG

### Testiranje značaja tačke rose za predikciju vlažnosti vazduha

In [ ]:
data = loadData("OutputData/WeatherAus_After_4_2_1_6.csv")
df = data.copy()

df['TackaRose9am'] = pd.to_numeric(df['TackaRose9am'], errors='coerce')
df['TackaRose3pm'] = pd.to_numeric(df['TackaRose3pm'], errors='coerce')
df['Temp9am'] = pd.to_numeric(df['Temp9am'], errors='coerce')
df['Temp3pm'] = pd.to_numeric(df['Temp3pm'], errors='coerce')
df['Humidity9am'] = pd.to_numeric(df['Humidity9am'], errors='coerce')
df['Humidity3pm'] = pd.to_numeric(df['Humidity3pm'], errors='coerce')

# Funkcija koja primenjuje Magnus-Tetensovu formulu
def izracunaj_vlaznost(temp, tacka_rose):
    # e_s = pritisak zasićene pare, e = stvarni pritisak pare
    e = np.exp((17.625 * tacka_rose) / (243.04 + tacka_rose))
    e_s = np.exp((17.625 * temp) / (243.04 + temp))

    rh = 100 * (e / e_s)

    # Ograničavamo maksimalnu vlažnost na 100% u slučaju malih grešaka senzora
    rh = np.clip(rh, 0, 100)
    return rh

# 2. Primena formule na podatke za 9am
mask_9am = df['Temp9am'].notna() & df['TackaRose9am'].notna() & df['Humidity9am'].notna()
df.loc[mask_9am, 'Calc_Humidity9am'] = izracunaj_vlaznost(df.loc[mask_9am, 'Temp9am'], df.loc[mask_9am, 'TackaRose9am'])

# 3. Primena formule na podatke za 3pm
mask_3pm = df['Temp3pm'].notna() & df['TackaRose3pm'].notna() & df['Humidity3pm'].notna()
df.loc[mask_3pm, 'Calc_Humidity3pm'] = izracunaj_vlaznost(df.loc[mask_3pm, 'Temp3pm'], df.loc[mask_3pm, 'TackaRose3pm'])

# 4. Izračunavanje metrika greške
print("--- Rezultati testiranja za 9 AM ---")
if mask_9am.sum() > 0:
    mae_9am = np.abs(df.loc[mask_9am, 'Humidity9am'] - df.loc[mask_9am, 'Calc_Humidity9am']).mean()
    kor_9am = df.loc[mask_9am, 'Humidity9am'].corr(df.loc[mask_9am, 'Calc_Humidity9am'])
    print(f"Uzorak: {mask_9am.sum()} redova")
    print(f"Srednja apsolutna greška (MAE): {mae_9am:.2f} %")
    print(f"Pirsonova korelacija: {kor_9am:.4f}")
else:
    print("Nema dovoljno preklapanja podataka za 9am.")

print("\n--- Rezultati testiranja za 3 PM ---")
if mask_3pm.sum() > 0:
    mae_3pm = np.abs(df.loc[mask_3pm, 'Humidity3pm'] - df.loc[mask_3pm, 'Calc_Humidity3pm']).mean()
    kor_3pm = df.loc[mask_3pm, 'Humidity3pm'].corr(df.loc[mask_3pm, 'Calc_Humidity3pm'])
    print(f"Uzorak: {mask_3pm.sum()} redova")
    print(f"Srednja apsolutna greška (MAE): {mae_3pm:.2f} %")
    print(f"Pirsonova korelacija: {kor_3pm:.4f}")
else:
    print("Nema dovoljno preklapanja podataka za 3pm.")

### Šta smo probali, a nije otišlo u produkciju


U fazi obogaćivanja podataka i inženjeringa obeležja (feature engineering), testirali smo mnoštvo kreativnih hipoteza u pokušaju da obezbedimo prednost našem modelu. Iako su skripte i algoritmi tehnički bili uspešno implementirani, sledeći pristupi nisu ušli u finalni produkcioni pajplajn jer su unosili previše šuma, bili računski neopravdano zahtevni ili prosto nisu doneli očekivan pad greške (RMSE).

Koncept "Klimatske udaljenosti" i analiza ruta između stanica 

Klasična geografska udaljenost nam se činila previše prostom, pa smo pokušali da kreiramo matricu takozvane klimatske udaljenosti između stanica. Ideja je bila da dve stanice koje su blizu, ali ih razdvaja visoka planina ili velika vodena površina, klimatski zapravo nisu "blizu". Da bismo ovo modelovali, prvo smo morali da analiziramo tipove terena. Korišćenjem biblioteke itertools.combinations, generisali smo sve moguće jedinstvene parove meteoroloških stanica i slali zahteve na Google Earth Engine kako bismo izvukli podatke iz ESA WorldCover satelitske mape. Za svaku pravolinijsku rutu između dve stanice kreirali smo pojas (buffer) širine 20 metara i prebrojavali piksele u rezoluciji od 10 metara za različite klase (npr. '10' za Drveće/Šumu, '30' za Travnjak, '50' za Beton, '80' za Vodu).

Nakon toga smo kreirali specifičnu formulu koja modifikuje baznu udaljenost: Klimatska_Distanca = Udaljenost (km) * (1 + alpha * Normalizovana_Visina) * (1 + beta * Indeks_Pokrivaca). Indeks pokrivača smo razvili na osnovu takozvanih Bowen-ovih pondera, množeći procente vlažnih površina sa 0.3, kopnene vegetacije sa 1.0, dok su suve i veštačke površine znatno penalizovane množenjem sa 5.0. Kako ovolike vrednosti ne bi "ugušile" uticaj visine (Alpha), ovaj indeks je dodatno skaliran pomoću MinMaxScaler-a. Zatim smo pokrenuli "grid search" pretragu (varirajući parametre alpha i beta od 0 do 3.5 sa korakom 0.5) i testirali model na 100 nasumičnih datuma, pokušavajući da predvidimo minimalnu temperaturu (MinTemp) koristeći proseke ponderisane udaljenošću K=3 najbliža suseda.

Zašto smo odustali: Zahtevi za procesorskom snagom bili su apsolutno ogromni, jer je Google API morao da obradi frekvencijske histograme na svim mogućim kombinacijama ruta. Tokom optimizacije pokazalo se da je pronalazak idealnih vrednosti za Alpha i Beta previše nestabilan i sklon "overfitting-u" na izabranom uzorku datuma. Dodavanje ovolike kompleksnosti nije donelo drastično obaranje RMSE metrike da bi opravdalo ogromno vreme izvršavanja.

Predviđanje oblačnosti pomoću "pada" satelitskog NDVI indeksa 

Kako bismo dopunili nedostajuće vrednosti za meteorološku oblačnost u 15 časova (Cloud3pm), testirali smo kreativnu ideju baziranu na optičkim senzorima. Iz MODIS (MOD09GQ) satelita povlačili smo dnevni vegetacioni indeks (NDVI) u visokoj rezoluciji od 250 metara. Zatim smo izračunali "14-dnevni pokretni maksimum" za svaku lokaciju, čime smo dobili stabilnu krivu vegetacije otpornu na oblake. Postavili smo pravilo: ukoliko dnevna vrednost NDVI indeksa padne za 0.1 ili više u odnosu na taj maksimum, algoritam beleži da je tog dana bilo značajne oblačnosti.

Zašto smo odustali: Ovaj sintetički prediktor smo uporedili sa stvarnim merenjima sa stanica (gde oblačnost >= 6 oktasa znači pretežno oblačan dan). Evaluacija je uključivala generisanje matrice konfuzije i računanje tačnosti, preciznosti i odziva. Računali smo i Pearsonovu korelaciju između pada NDVI-a i pravog broja oktasa. Pokazalo se da je senzor beležio ogroman broj lažno oblačnih dana (NDVI je pao, a bilo je vedro) jer vlaga u vazduhu ili prolazna kiša drastično utiču na refrakciju svetlosti, a bilo je i mnogo lažno vedrih dana, što je srušilo ukupnu tačnost modela.

"Coastline Fix" - Spiralna pretraga obale za vlažnost tla Želeli smo da obogatimo podatke vlažnošću površinskog sloja tla (0-7cm dubine), te smo koristili ERA5-Land satelitski model. Veliki problem bio je taj što je veliki broj meteoroloških stanica u Australiji smešten na samoj obali, zbog čega je model (koji detektuje te piksele kao okean) stalno vraćao NaN vrednosti. Napisali smo "Coastline Fix" skriptu koja je za problematične stanice radila spiralnu pretragu koristeći matricu pomaka u stepenima. Pomoću Haversine formule, skripta je pretraživala susedne piksele na udaljenosti od ~11 km, zatim po dijagonali na ~15 km, i najzad širi krug na ~22 km, sve dok ne pronađe prvi validan "kopneni" piksel i zabeleži tačnu vazdušnu udaljenost pomaka.

Zašto smo odustali: Nakon vizualizacije izvezene kolone Udaljenost_pomeraja_km, uvideli smo da je pozajmljivanje vrednosti vlage sa lokacija pomerenih preko 20 kilometara ka unutrašnjosti suviše veštačko. Vlažnost tla je izuzetno mikrolokalna, podložna brzom isušivanju od vetra i specifičnosti zemljišta, pa je preuzimanje ovih vrednosti činilo više štete nego koristi za analizu obalnih gradova.

Visokorezolucijski Topografski Pozicioni Indeks (TPI) 

Smatrali smo da makro reljef (nadmorska visina iz uobičajenih setova podataka) nije dovoljan i da model mora da razume da li je stanica u mikrouvali ili na padini. Koristili smo OpenTopoData API (SRTM30m model) za skidanje visina i kreirali matricu od 3x3 tačke (uz ofset od 0.005 stepeni) oko centralne stanice. TPI je zatim računat jednostavnim oduzimanjem prosečne visine svih validnih tačaka iz matrice od visine centralne tačke.

Zašto smo odustali: Prvo, proces je bio bolno spor jer smo zbog striktnih restrikcija API servera morali da forsiramo programsku pauzu od 1.1 sekunde (time.sleep(1.1)) između poziva za svaku stanicu. Drugo, inženjerski gledano, ispostavilo se da topografsko odstupanje na tako finoj distanci prosto ne nosi nikakvu statističku težinu kada pokušavate da predvidite masivna frontalna pomeranja kiše.

Kombinovanje hibridnih dugoročnih klimatskih indeksa (ENSO i SAM) 

Kao ultimativni test, probali smo da integrišemo makro-klimatske fenomene u dnevne podatke. Učitali smo bazu ENSO vrednosti još od 1948. godine, izračunali mesečnu klimatologiju i generisali anomalije oduzimanjem proseka. Ograničili smo vizuelizaciju isključivo na istok i severoistok (Longituda > 135, Latituda < -10). Zatim smo jednostavno sabrali ENSO anomaliju i sekundarni indeks ("Index2") u hibridni "Combined_Index" i plotovali ga naspram prosečnog dnevnog vazdušnog pritiska (srednja vrednost pritiska u 9 i 15 časova) za dane kada je potvrđeno da će padati kiša sutradan (RainTomorrow == 'Yes').

Zašto smo odustali: Rezultati na raspršenim (scatter) grafikonima pokazali su ogromno preklapanje tačaka bez obzira na boju klasa (da li će padati kiša ili ne). Iako su ENSO i SAM presudni za sezonsku prognozu (koliko će padavina biti u naredna 3 meseca), njihov signal na skali od 24 časa je jednostavno preslab i potpuno prekriven lokalnim dnevnim šumom, zbog čega ih model mašinskog učenja uglavnom ignoriše kao obeležja.